# Import des librairies

In [1]:
import pandas as pnd

pnd.set_option('display.max_columns', None)  # Show all columns
pnd.set_option('future.no_silent_downcasting', True)

/tmp/ipykernel_21134/1276126716.py:4: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pnd.set_option('future.no_silent_downcasting', True)


# Découpage des observations en jeu d'apprentissage et jeu de tests

In [2]:
#Chargement du dataset avec pour séparateur des tabulations
dataset = pnd.read_csv("datas/dataset.csv",delimiter='\t')

#Suppression des lignes ayant des valeurs manquantes
dataset = dataset.dropna(axis=0, how='any')

#X = on prend toutes les données, mais uniquement les features 5 à 12 
#(ne pas hésiter à ouvrir le fichier afin de bien visualiser les features concernées)
#POINTS_ATTAQUE;POINTS_DEFFENCE;POINTS_ATTAQUE_SPECIALE;POINT_DEFENSE_SPECIALE;POINTS_VITESSE;NOMBRE_GENERATIONS
X = dataset.iloc[:, 5:12].values

#y = on prend uniquement la feature POURCENTAGE_DE_VICTOIRE (17ème feature)
y = dataset.iloc[:, 17].values

#X = on prend toutes les données, mais uniquement les features 4 à 11
# POINTS_ATTAQUE;POINTS_DEFFENCE;POINTS_ATTAQUE_SPECIALE;POINT_DEFENSE_SPECIALE;POINTS_VITESSE;NOMBRE_GENERATIONS
X = dataset.iloc[:, 5:12].values

#y = on prend uniquement la colonne POURCENTAGE_DE_VICTOIRE (16ème feature) 
#les : signifiant "Pour toutes les observations"
y = dataset.iloc[:, 16].values

#Construction du jeu d'entrainement et du jeu de tests
from sklearn.model_selection import train_test_split
X_APPRENTISSAGE, X_VALIDATION, Y_APPRENTISSAGE, Y_VALIDATION = train_test_split(X, y, test_size = 0.2, random_state = 0)

# Algorithme de régression de linéaire

In [3]:
#---- ALGORITHME 1: REGRESSION LINEAIRE -----
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression

#Choix de l'algorithme
algorithme = LinearRegression()

#Apprentissage à l'aide de la fonction fit
#Apprentisage effectué sur le jeu de données d'apprentissage
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)

#Realisation de prédictions sur le jeu de tests (validation)
predictions = algorithme.predict(X_VALIDATION)

#Calcul de la précision de l'apprentissage à l'aide de la
#fonction r2_score en comparant les valeurs prédites
#(predictions) et les valeurs attendues (Y_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- REGRESSION LINEAIRE -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- REGRESSION LINEAIRE -----------
>> Precision = 0.8828844617689129
------------------------------------------


# Algorithme des forêts aléatoires

In [4]:
#Choix de l'algorithme
from sklearn.ensemble import RandomForestRegressor
algorithme = RandomForestRegressor()
algorithme.fit(X_APPRENTISSAGE, Y_APPRENTISSAGE)
predictions = algorithme.predict(X_VALIDATION)
precision = r2_score(Y_VALIDATION, predictions)

print(">> ----------- FORETS ALEATOIRES -----------")
print(">> Precision = "+str(precision))
print("------------------------------------------")

>> ----------- FORETS ALEATOIRES -----------
>> Precision = 0.896395487154043
------------------------------------------


# Sauvegarde du modèle d'apprentissage

In [5]:
import joblib
fichier = 'modele/modele_pokemon.mod'
joblib.dump(algorithme, fichier)

['modele/modele_pokemon.mod']

# Phénomènes de surapprentissage (overfitting) et de sous-apprentissage (underfitting)

Lorsque l’on cherche une solution à un problème en Machine Learning, nous souhaitons que celle-ci soit généralisable. C’est-à-dire que la solution trouvée doit être applicable sur des données inconnues de celles utilisées lors de l’apprentissage.

Pour cela, il faut que les données utilisées lors de la phase d’apprentissage soient les plus proches possible de la réalité et du problème à résoudre.

Le **surapprentissage (overfitting)** est un phénomène se traduisant par le fait que la solution est trop adaptée aux données d’apprentissage et ne se généralise pas à de nouvelles données qui lui sont inconnues. Ainsi, si pour un algorithme nous obtenons une précision 99 % sur les données d’apprentissage et que nous obtenons une valeur de 20 % sur les données de tests, il y fort à parier que nous sommes en présence d’un surapprentissage. C’est pourquoi il est conseillé de mesurer la précision à la fois sur les données d’apprentissage et sur les données de validation :

In [6]:
predictions = algorithme.predict(X_VALIDATION)
precision_apprentissage = algorithme.score(X_APPRENTISSAGE,Y_APPRENTISSAGE)
precision = r2_score(Y_VALIDATION, predictions)

Quant au phénomène de **sous-apprentissage**, ce dernier se produit lorsque l’algorithme n’arrive pas à trouver une corrélation entre les données d’apprentissage et n’arrive donc pas à réaliser de bonnes prédictions.

La notion d’overfitting est souvent rencontrée lors de l’utilisation des réseaux de neurones. En effet, lors de l’apprentissage nous allons chercher à minimiser les erreurs de prédictions en réalisant un certain nombre de boucles d’apprentissage (itérations) où, à chaque boucle, l’algorithme va apprendre de ses erreurs et se corriger.

Plus nous réalisons d’itérations, plus l’algorithme apprend et moins il se trompe. Si trop d’itérations sont réalisées, l’algorithme aura une très bonne précision sur les données d’apprentissage, mais il se sera spécialisé sur celles-ci et aura une mauvaise précision sur les données de tests. La généralisation n’est donc plus possible.